[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barmag/fanous-llm-lens/blob/main/notebooks/education/probe_a_linear_reference.ipynb)

# Rung `probe_a` — what *is* a probe?  ·  ما هو المِسبار؟

A **probe** is the simplest interpretability tool: train a tiny classifier (here,
logistic regression) on a model's internal activations and ask *can it read a
property back out?* If a linear probe recovers a feature, that feature is
**linearly present** in the representation.

Probing has a famous trap, and this rung is built around it. Alain & Bengio (2016)
introduced linear probes; Hewitt & Liang (2019) then showed a probe can score high
by **memorising**, and Belinkov (2022) surveyed the deeper issue: *a probe measures
the input encoding as much as the network.* A high score does **not** prove the model
computed anything.

To see that trap in its cleanest form, we probe a **zero-layer** model — embeddings +
positional + LayerNorm + a tied head, with **no attention and no MLP**. Every scrap of
learnable structure lives in the embedding table `W_E`, so a probe on its pooled
embeddings measures what the **tokenization** makes available, and nothing the layers
"understood" — because there are no layers.

> **المِسبار** يدرِّب مُصنِّفًا بسيطًا على التمثيل الداخلي للنموذج ليسأل: هل هذه الخاصية
> قابلة للاستخراج خطيًّا؟ سنستخدم نموذجًا **بلا طبقات** (تضمين فقط) لنُظهر أن درجةً عالية
> للمِسبار قد تعكس **التقطيع** لا ما تعلّمه النموذج.

**What this rung can show:** the probing workflow end-to-end (extract → pool → fit →
read AUC), the two controls every probe needs, and the decisive test — *does training
move the probe?*

**What it cannot show:** anything about a *learned* representation. A zero-layer model's
contribution to the probe is ≈0 by construction. That question — does computation ever
show up in a probe? — is `probe_c`'s, and the honest answer there is already on record
(depth alone did not manufacture a signal; the levers are readout and task pressure).

## Setup

We fetch two shared helpers on Colab: `tiny.py` (device pick + the compact mGPT
encoder the ladder uses) and `probes.py` (the zero-layer model, pooling, and the
probe + controls). Locally they import as sibling modules.

In [1]:
# 🛠️ Setup: install deps + fetch the shared helpers on Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformer_lens datasets transformers scikit-learn plotly
    !pip uninstall -y -q torchaudio
    !wget -q https://raw.githubusercontent.com/barmag/fanous-llm-lens/main/notebooks/education/tiny.py
    !wget -q https://raw.githubusercontent.com/barmag/fanous-llm-lens/main/notebooks/education/probes.py
import numpy as np
import torch
import plotly.graph_objects as go
from IPython.display import HTML, display
import tiny
import probes

torch.manual_seed(0)
DEVICE = tiny.device()
print("device:", DEVICE)

# Knobs. verify_notebooks.py injects tiny CI values (and FORCE_SYNTHETIC) via globals().
FORCE_SYNTHETIC = globals().get("FORCE_SYNTHETIC", False)
MAX_MSA   = globals().get("MAX_MSA", 8000)
MAX_MASRI = globals().get("MAX_MASRI", 4000)
MAX_STEPS = globals().get("MAX_STEPS", 2000)
D_MODEL   = globals().get("D_MODEL", 256)

device: cuda


/home/yassermakram/code/fanous-llm-lens/notebooks/education/tiny.py:25: UserWarning: expandable_segments not supported on this platform (Triggered internally at ../c10/hip/HIPAllocatorConfig.h:29.)
  torch.ones(1, device="cuda").add_(1)


## Step 1 — look at 5 rows first

Before pulling a full corpus we look at a handful of rows: confirm the two registers,
the labels, and that Arabic renders right-to-left. MSA comes from Arabic Wikipedia,
Masri from Egyptian-dialect tweets. **Masri is label 1** — the register the project
cares about.

In [2]:
# Step 1 — sample the corpus.
if FORCE_SYNTHETIC:
    data = probes.synthetic_dialect_corpus(n_per_reg=200)
else:
    data = probes.load_dialect_corpus(max_msa=MAX_MSA, max_masri=MAX_MASRI)

sentences, register = data["sentences"], data["register"]
n_msa, n_masri = int((register == 0).sum()), int((register == 1).sum())
print(f"{len(sentences)} sentences | MSA {n_msa} | Masri {n_masri}")

def show_rtl(rows, title):
    body = "".join(f"<p style='margin:.2em 0'>{r}</p>" for r in rows)
    display(HTML(f"<b>{title}</b><div dir='rtl' style='text-align:right;font-size:1.05em'>{body}</div>"))

show_rtl([s for s, r in zip(sentences, register) if r == 0][:5], "MSA — 5 rows")
show_rtl([s for s, r in zip(sentences, register) if r == 1][:5], "Masri — 5 rows")

12000 sentences | MSA 8000 | Masri 4000


## Step 2 — a compact encoder and the zero-layer model

We tokenise with the pretrained **mGPT** tokenizer (Arabic-aware) but keep only the
subword ids that actually appear, remapped to a compact 0..K space — a 100k-row
embedding won't train on the iGPU, a few-thousand-row one will. Then we build the
**zero-layer model** and train it just enough to *reshape* `W_E`; we don't need a
strong language model, only a trained embedding table to probe.

In [3]:
# Step 2 — build the encoder, then train the zero-layer model.
if FORCE_SYNTHETIC:
    encode, d_vocab = probes.whitespace_encoder(sentences)
else:
    from transformers import AutoTokenizer
    mgpt = AutoTokenizer.from_pretrained("ai-forever/mGPT")
    encode, _ids, _id2str, d_vocab = tiny.make_compact_encoder(mgpt, sentences)
print("compact vocab size:", d_vocab)

stream = []
for s in sentences:
    stream += encode(s)
print("token-stream length:", len(stream))

cfg = probes.ZeroLayerConfig(vocab_size=d_vocab, d_model=D_MODEL, seq_len=64,
                             batch_size=64, max_steps=MAX_STEPS, device=DEVICE)
result = probes.train_zero_layer(cfg, stream, log_every=max(1, MAX_STEPS // 4))
print("final loss:", round(result["final_loss"], 3))

fig = go.Figure(go.Scatter(y=result["losses"], mode="lines"))
fig.update_layout(title="Zero-layer training loss", xaxis_title="step", yaxis_title="loss",
                  height=320, template="plotly_white")
fig.show()

compact vocab size: 9125
token-stream length: 947153


/home/yassermakram/code/fanous-llm-lens/.venv/lib64/python3.11/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at ../aten/src/ATen/Context.cpp:296.)
  return F.linear(input, self.weight, self.bias)


  step   500/2000  loss=56.797
  step  1000/2000  loss=15.155
  step  1500/2000  loss=7.314
  step  2000/2000  loss=7.177
final loss: 7.177


## Step 3 — the probe

Represent each sentence as the **mean of its token embeddings** (pooling — uniform
across tokenizers), then fit logistic regression to predict the register on a held-out
split, scored by ROC-AUC. Two controls run alongside:

- **`random`** (coin-flip labels) must sit at **AUC ≈ 0.50** — proof of no leakage.
- **`length`** (longer than the median?) must be **decodable** (AUC > 0.50) — proof the
  probe and representation actually work.

A probe you don't trust without both controls green.

In [4]:
# Step 3 — pool, fit, read AUC (with controls).
w_e = result["model"].w_e
X = probes.pooled_embeddings(w_e, encode, sentences)
print("pooled matrix:", X.shape)

controls = probes.make_controls(sentences)
auc_dialect = probes.probe_auc(X, register)
auc_random  = probes.probe_auc(X, controls["random"])
auc_length  = probes.probe_auc(X, controls["length"])
print(f"AUC  dialect={auc_dialect}   random={auc_random} (want ~0.50)   length={auc_length} (want >0.50)")

pooled matrix: (12000, 256)
AUC  dialect=0.981   random=0.505 (want ~0.50)   length=0.816 (want >0.50)


## Step 4 — the decisive control, and the calibration gate

Dialect will likely score **very high** — Masri tweets and MSA Wikipedia use largely
*different words*, so the pooled bag-of-tokens is nearly separable on vocabulary alone.
Before crediting that to the model, run the control that decides everything: **probe a
fresh, untrained model**. If training barely moves the AUC, the probe is reading the
*tokenization*, not anything the model learned.

This is also our **calibration gate**: if dialect clears the control floor (it will),
it becomes the ladder's through-line feature, revisited with depth at `probe_c`. If a
feature *didn't* clear the floor on this data + model, that would be the honest finding,
and we'd say so rather than force a result.

In [5]:
# Step 4 — does *training* move the probe?
untrained = probes.ZeroLayerModel(d_vocab, D_MODEL, max_len=64)
X_untrained = probes.pooled_embeddings(untrained.w_e, encode, sentences)
auc_untrained = probes.probe_auc(X_untrained, register)
delta = round((auc_dialect or 0) - (auc_untrained or 0), 3)
print(f"dialect AUC   trained={auc_dialect}   untrained={auc_untrained}   Δ(trained−untrained)={delta}")

dialect AUC   trained=0.981   untrained=0.984   Δ(trained−untrained)=-0.003


## The lesson

The probe scored near-perfect on dialect — and the **untrained** model scored just as
high. Training reshaped `W_E` (the loss fell), yet the probe barely moved. So the honest
reading is **not** "the model understands dialect." It is:

> At zero-layer depth, a pooled linear probe scores the **tokenization**, not the model.
> A high AUC can come entirely from the input encoding.

> سجّل المِسبار درجةً شبه كاملة على اللهجة — والنموذج **غير المُدرَّب** سجّل مثلها. أي أن
> الدرجة تعكس **التقطيع** لا ما تعلّمه النموذج.

This is the trap Hewitt & Liang and Belinkov warned about, made concrete: no computation
happened, yet the probe reads 1.0. Every probing result you meet should trigger the same
reflex — *compare against a control that trained on nothing.*

In [6]:
# A second, word-level probe: definiteness — does the word start with the article ال?
# (A gold-free teaching proxy; real work would use camel-tools morphology.)
words = sorted({w for s in sentences for w in s.split() if len(w) >= 2})
y_def = probes.definite_labels(words)
Xw = probes.pooled_embeddings(w_e, encode, words)
print(f"words={len(words)}  definite +{int(y_def.sum())}")
print("definiteness AUC:", probes.probe_auc(Xw, y_def), "(None = too few types on this small corpus)")

words=99487  definite +19730
definiteness AUC: 0.962 (None = too few types on this small corpus)


## Recap & handoff

- A **probe** = a simple classifier on activations; AUC = linear recoverability.
- Every probe needs **controls**: `random` ≈ 0.50 (no leakage) and `length` decodable
  (probe works).
- The **decisive control** is trained vs **untrained**: here they match, so the score is
  a property of the **tokenization**, not the model.

**Next — `probe_b` (can you trust it?):** the rigor rung. We make the trained−untrained
increment the headline quantity and add **selectivity / control tasks** (Hewitt & Liang)
and a **type-split**, so a memorising probe can't fool us.

**Then — `probe_c` (where does it live?):** add depth and read the residual stream
layer-by-layer — and meet the honest null that depth alone does *not* manufacture a
learned signal (the levers are readout and task pressure), so we learn what actually
makes computation visible to a probe.